In [ ]:
import kagglehub
import pandas as pd
import numpy as np
import joblib
import os

# Get the path to the dataset
path = str(kagglehub.dataset_download("moltean/fruits"))
print("Path to dataset files:", path)

def create_csv(dataset_type):
    dataset_path = f'{path}/fruits-360_100x100/fruits-360/{dataset_type}'
    image_paths = []
    labels = []

    for label in os.listdir(dataset_path):
        label_path = os.path.join(dataset_path, label)
        if os.path.isdir(label_path):
            for image_name in os.listdir(label_path):
                if image_name.endswith('.jpg'):
                    image_paths.append(os.path.join(label, image_name))
                    labels.append(label)

    df = pd.DataFrame({
        'image': image_paths,
        'label': labels
    })

    csv_filename = f'fruits_{dataset_type.lower()}.csv'
    df.to_csv(csv_filename, index=False)
    print(f"{dataset_type} CSV saved at: {os.path.join(os.getcwd(), csv_filename)}")

# Create CSVs for training and testing datasets
create_csv('Training')
create_csv('Test')

train_csv = os.path.join(os.getcwd(), 'fruits_training.csv')
test_csv = os.path.join(os.getcwd(), 'fruits_test.csv')

# Load them into DataFrames
train_df = pd.read_csv(train_csv)
test_df = pd.read_csv(test_csv)

# Check the first few rows
print(train_df.head())
print(test_df.head())

def limit_classes(df, num_classes=50, samples_per_class=None, random_state=42):
    # Pick a subset of classes
    unique_classes = df["label"].unique()
    np.random.seed(random_state)
    selected_classes = np.random.choice(unique_classes, num_classes, replace=False)

    # Filter down to those classes
    df = df[df["label"].isin(selected_classes)]

    # Optionally, sample equal images per class
    if samples_per_class:
        df = df.groupby("label").apply(
            lambda x: x.sample(n=min(samples_per_class, len(x)), random_state=random_state)
        ).reset_index(drop=True)

    return df




imgPerClass = 50

#train_df = limit_classes(train_df, samples_per_class=imgPerClass)
#test_df = limit_classes(test_df, samples_per_class=imgPerClass)

In [ ]:
from sklearn.metrics import accuracy_score, classification_report
from concurrent.futures import ProcessPoolExecutor
from sklearn.preprocessing import LabelEncoder
from functools import partial
from PIL import Image
import numpy as np
import os

size = (64, 64)

def process_image(row, base_path, crop_size):
    image_path = os.path.join(base_path, row['image'])
    img = Image.open(image_path).convert('RGB')
    img = img.resize(crop_size)
    # use float32 for smaller memory footprint
    img_array = np.array(img, dtype=np.float32) / 255.0
    label = row['label']
    return img_array, label


def load_parallel(csv_df, base_path, crop_size=size, n_jobs=30):
    rows = csv_df.to_dict("records")
    n_samples = len(rows)
    
    # pre-allocate big array once
    X = np.empty((n_samples, crop_size[0], crop_size[1], 3), dtype=np.float32)
    y = [None] * n_samples

    with ProcessPoolExecutor(max_workers=n_jobs) as executor:
        func = partial(process_image, base_path=base_path, crop_size=crop_size)
        for i, (img_array, label) in enumerate(executor.map(func, rows, chunksize=50)):
            X[i] = img_array
            y[i] = label

    return X, np.array(y)


# usage
base_path_train = os.path.join(path, "fruits-360_100x100/fruits-360/Training")
X_train, y_train = load_parallel(train_df, base_path_train)

base_path_val = os.path.join(path, "fruits-360_100x100/fruits-360/Test")
X_val, y_val = load_parallel(test_df, base_path_val)

print("Train:", X_train.shape, "Val:", X_val.shape)
print("Unique train labels:", np.unique(y_train))
print("Unique val labels:", np.unique(y_val))

# flatten for models that don’t support images directly
X_train_flat = X_train.reshape(X_train.shape[0], -1)
X_val_flat   = X_val.reshape(X_val.shape[0], -1)

# encode labels
le = LabelEncoder()
y_train_enc = le.fit_transform(y_train)
y_val_enc   = le.transform(y_val)

print(y_train_enc[:20])
print(y_val_enc[:20])


In [ ]:
import matplotlib.pyplot as plt
import random
from PIL import Image
import os

# Pick 5 random rows
sample_rows = train_df.sample(5, random_state=42)

# Set base path
base_path_train = os.path.join(path, "fruits-360_100x100/fruits-360/Training")

# Create figure
plt.figure(figsize=(15, 5))

for i, (_, row) in enumerate(sample_rows.iterrows(), 1):
    img_path = os.path.join(base_path_train, row['image'])
    img = Image.open(img_path).convert('RGB')
    
    plt.subplot(1, 5, i)
    plt.imshow(img)
    plt.title(row['label'])
    plt.axis('off')

plt.show()
